In [0]:

# Create Silver schema

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

print("Silver schema created successfully!")


Silver schema created successfully!


In [0]:

# Read Bronze tables

customers_df = spark.table("workspace.bronze.customers_raw")

orders_df = spark.table("workspace.bronze.orders_raw")

products_df = spark.table("workspace.bronze.products_raw")

order_items_df = spark.table("workspace.bronze.order_items_raw")

payments_df = spark.table("workspace.bronze.payments_raw")

reviews_df = spark.table("workspace.bronze.reviews_raw")

sellers_df = spark.table("workspace.bronze.sellers_raw")

geo_df = spark.table("workspace.bronze.geolocation_raw")

category_df = spark.table("workspace.bronze.category_translation_raw")

print("All nine Bronze tables loaded successfully!")


All nine Bronze tables loaded successfully!


In [0]:

from pyspark.sql.functions import col

# Remove duplicate orders using order_id
orders_clean = orders_df.dropDuplicates(["order_id"])

# Check missing values in important columns
print("Total orders before cleaning:", orders_df.count())

print("Total orders after removing duplicates:", orders_clean.count())

print("Missing order IDs:",
      orders_clean.filter(col("order_id").isNull()).count())

print("Missing customer IDs:",
      orders_clean.filter(col("customer_id").isNull()).count())

# Save cleaned data
(orders_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.orders_clean"))

print("Silver orders table created successfully!")


Total orders before cleaning: 99441
Total orders after removing duplicates: 99441
Missing order IDs: 0
Missing customer IDs: 0
Silver orders table created successfully!


In [0]:

from pyspark.sql.functions import col

# 1. Clean Customers
customers_clean = customers_df.dropDuplicates(["customer_id"])

# 2. Clean Geolocation
geo_clean = geo_df.dropDuplicates()

# 3. Clean Order Items
order_items_clean = order_items_df.dropDuplicates(
    ["order_id", "order_item_id"]
)

# 4. Clean Payments
payments_clean = payments_df.dropDuplicates()

# 5. Clean Reviews
reviews_clean = reviews_df.dropDuplicates(["review_id"])

# 6. Clean Products
products_clean = products_df.dropDuplicates(["product_id"])

# 7. Clean Sellers
sellers_clean = sellers_df.dropDuplicates(["seller_id"])

# 8. Clean Category Translation
category_clean = category_df.dropDuplicates(
    ["product_category_name"]
)

print("All eight datasets cleaned successfully!")


All eight datasets cleaned successfully!


In [0]:

# Dictionary of cleaned DataFrames
silver_tables = {
    "customers_clean": customers_clean,
    "geolocation_clean": geo_clean,
    "order_items_clean": order_items_clean,
    "payments_clean": payments_clean,
    "reviews_clean": reviews_clean,
    "products_clean": products_clean,
    "sellers_clean": sellers_clean,
    "category_translation_clean": category_clean
}

# Save each DataFrame as a Silver Delta table
for table_name, df in silver_tables.items():

    (df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable("workspace.silver." + table_name))

    print(table_name, "created successfully")

print("ALL SILVER TABLES SAVED SUCCESSFULLY!")


customers_clean created successfully
geolocation_clean created successfully
order_items_clean created successfully
payments_clean created successfully
reviews_clean created successfully
products_clean created successfully
sellers_clean created successfully
category_translation_clean created successfully
ALL SILVER TABLES SAVED SUCCESSFULLY!


In [0]:

# Display all Silver tables

silver_list = spark.sql("SHOW TABLES IN workspace.silver")

display(silver_list)

print("Total Silver tables:", silver_list.count())


database,tableName,isTemporary
silver,category_translation_clean,false
silver,customers_clean,false
silver,geolocation_clean,false
silver,order_items_clean,false
silver,orders_clean,false
silver,payments_clean,false
silver,products_clean,false
silver,reviews_clean,false
silver,sellers_clean,false


Total Silver tables: 9


In [0]:

from pyspark.sql.functions import col

# Load the Silver tables
customers = spark.table("workspace.silver.customers_clean")
orders = spark.table("workspace.silver.orders_clean")
items = spark.table("workspace.silver.order_items_clean")
products = spark.table("workspace.silver.products_clean")
sellers = spark.table("workspace.silver.sellers_clean")
payments = spark.table("workspace.silver.payments_clean")
reviews = spark.table("workspace.silver.reviews_clean")

# Function to find records with missing parent references
def check_relationship(child_df, child_key, parent_df, parent_key, name):

    child_keys = (
        child_df
        .select(col(child_key).alias("key"))
        .where(col(child_key).isNotNull())
        .distinct()
    )

    parent_keys = (
        parent_df
        .select(col(parent_key).alias("key"))
        .where(col(parent_key).isNotNull())
        .distinct()
    )

    unmatched = child_keys.join(parent_keys, "key", "left_anti")

    null_count = child_df.filter(col(child_key).isNull()).count()
    unmatched_count = unmatched.count()

    print("Relationship:", name)
    print("Null references:", null_count)
    print("Unmatched references:", unmatched_count)
    print("-----------------------------------")


# 1. Orders -> Customers
check_relationship(
    orders, "customer_id",
    customers, "customer_id",
    "Orders to Customers"
)

# 2. Order Items -> Orders
check_relationship(
    items, "order_id",
    orders, "order_id",
    "Order Items to Orders"
)

# 3. Order Items -> Products
check_relationship(
    items, "product_id",
    products, "product_id",
    "Order Items to Products"
)

# 4. Order Items -> Sellers
check_relationship(
    items, "seller_id",
    sellers, "seller_id",
    "Order Items to Sellers"
)

# 5. Payments -> Orders
check_relationship(
    payments, "order_id",
    orders, "order_id",
    "Payments to Orders"
)

# 6. Reviews -> Orders
check_relationship(
    reviews, "order_id",
    orders, "order_id",
    "Reviews to Orders"
)

print("RELATIONSHIP VALIDATION COMPLETED!")


Relationship: Orders to Customers
Null references: 0
Unmatched references: 0
-----------------------------------
Relationship: Order Items to Orders
Null references: 0
Unmatched references: 0
-----------------------------------
Relationship: Order Items to Products
Null references: 0
Unmatched references: 0
-----------------------------------
Relationship: Order Items to Sellers
Null references: 0
Unmatched references: 0
-----------------------------------
Relationship: Payments to Orders
Null references: 0
Unmatched references: 0
-----------------------------------
Relationship: Reviews to Orders
Null references: 2176
Unmatched references: 1032
-----------------------------------
RELATIONSHIP VALIDATION COMPLETED!


In [0]:

from pyspark.sql.functions import col

# Load Silver tables
reviews = spark.table("workspace.silver.reviews_clean")
orders = spark.table("workspace.silver.orders_clean")

# 1. Check total review records
print("Total reviews:", reviews.count())

# 2. Check reviews with missing order IDs
missing_reviews = reviews.filter(
    col("order_id").isNull()
)

print("Reviews with missing order IDs:",
      missing_reviews.count())

# 3. Display examples of missing order IDs
display(missing_reviews.limit(10))

# 4. Find reviews whose order IDs do not exist
# in the Orders table

unmatched_reviews = reviews.join(
    orders.select("order_id"),
    on="order_id",
    how="left_anti"
)

print("Unmatched review records:",
      unmatched_reviews.count())

# 5. Display examples
display(unmatched_reviews.limit(10))


Total reviews: 102958
Reviews with missing order IDs: 2176


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,ingestion_timestamp,source_file
Estou satisfeita,null,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
",2018-03-14 00:00:00,2018-03-15 11:57:51",null,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
Os produtos são de ótima qualidade.,null,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
e temos que confiar que o brasil tem jeito sim!,null,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
",2018-08-24 00:00:00,2018-08-27 11:46:00",null,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
Por exemplo,null,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
Na compra no baratheon agente compra hoje.,null,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
",2018-07-13 00:00:00,2018-07-15 17:42:01",null,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
",2018-07-22 00:00:00,2018-07-24 22:37:49",null,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
",2018-02-02 00:00:00,2018-02-03 13:40:49",null,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv


Unmatched review records: 4548


order_id,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,ingestion_timestamp,source_file
null,Estou satisfeita,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
null,",2018-03-14 00:00:00,2018-03-15 11:57:51",null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
2017-10-12 00:00:00,"Sensacional!!!""",2017-10-14 23:04:33,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
2018-07-27 00:00:00,"Péssimos atendimentos das lojas lannister e parceira!!!""",2018-07-30 16:41:46,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
não tenho como recomendar a loja.,Até que o produto seja entregue,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
2018-08-31 00:00:00,"Juliana""",2018-08-31 21:00:34,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
2017-12-29 00:00:00,"Obrigada""",2018-01-02 15:44:08,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
null,Os produtos são de ótima qualidade.,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
2017-04-02 00:00:00,"Não sei nem. O motivo """,2017-04-02 11:52:01,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv
null,e temos que confiar que o brasil tem jeito sim!,null,null,null,null,null,2026-11-16T06:16:27.880Z,olist_order_reviews_dataset.csv


In [0]:

# Define the raw data location
raw_path = "/Volumes/workspace/default/olist_raw_volume/"

# Read Reviews CSV with improved parsing settings
reviews_test = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(raw_path + "olist_order_reviews_dataset.csv")
)

print("Corrected Reviews Dataset")
print("-------------------------------")

# Check schema
reviews_test.printSchema()

# Count records
print("Total review records:", reviews_test.count())

# Display first 10 records
display(reviews_test.limit(10))


Corrected Reviews Dataset
-------------------------------
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

Total review records: 99224


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13T00:00:00.000Z,2018-04-16T00:39:37.000Z
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16T00:00:00.000Z,2017-07-18T19:30:34.000Z
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14T00:00:00.000Z,2018-08-14T21:36:06.000Z
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17T00:00:00.000Z,2017-05-18T12:05:37.000Z
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22T00:00:00.000Z,2018-05-23T16:45:47.000Z


In [0]:

# Replace incorrectly parsed Bronze Reviews table

(reviews_test.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.reviews_raw"))

print("Bronze Reviews table corrected successfully!")


Bronze Reviews table corrected successfully!


In [0]:

# Remove duplicate review IDs
reviews_clean = reviews_test.dropDuplicates(["review_id"])

# Save corrected Silver table
(reviews_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.reviews_clean"))

print("Silver Reviews table corrected successfully!")

print("Total corrected reviews:", reviews_clean.count())


Silver Reviews table corrected successfully!
Total corrected reviews: 98410


In [0]:

from pyspark.sql.functions import col

reviews = spark.table("workspace.silver.reviews_clean")
orders = spark.table("workspace.silver.orders_clean")

# Missing order IDs
missing = reviews.filter(
    col("order_id").isNull()
).count()

# Unmatched order IDs
unmatched = reviews.join(
    orders.select("order_id"),
    on="order_id",
    how="left_anti"
).count()

print("Total reviews:", reviews.count())
print("Missing order IDs:", missing)
print("Unmatched review records:", unmatched)


Total reviews: 98410
Missing order IDs: 0
Unmatched review records: 0


In [0]:

from pyspark.sql.functions import col, count, countDistinct

# Load corrected Bronze reviews
reviews_bronze = spark.table("workspace.bronze.reviews_raw")

# Find repeated review IDs
duplicate_ids = (
    reviews_bronze
    .groupBy("review_id")
    .agg(
        count("*").alias("record_count"),
        countDistinct("order_id").alias("different_orders")
    )
    .filter(col("record_count") > 1)
)

print("Duplicate review ID groups:", duplicate_ids.count())

print("Records with null review IDs:",
      reviews_bronze.filter(
          col("review_id").isNull()
      ).count())

display(duplicate_ids.limit(10))


Duplicate review ID groups: 789
Records with null review IDs: 0


review_id,record_count,different_orders
28642ce6250b94cc72bc85960aec6c62,2,2
a0a641414ff718ca079b3967ef5c2495,2,2
f4d74b17cd63ee35efa82cd2567de911,2,2
ecbaf1fce7d2c09bfab46f89065afeaf,2,2
6b1de94de0f4bd84dfc4136818242faa,2,2
957011305e7a4b6c8a266eeeb8e0316d,2,2
b34f4a786bf00b7c8486141ba482783b,2,2
d433c252647c51309432ca0b763f969b,2,2
5f35bbde2b32b8617aab15b7a9a0ae24,2,2
c5976a5a98e854fb23d7e03c6754ae60,2,2


In [0]:

# Find records having repeated review IDs

duplicate_review_records = reviews_bronze.join(
    duplicate_ids.select("review_id"),
    on="review_id",
    how="inner"
)

# Display the actual records
display(
    duplicate_review_records.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date"
    ).orderBy("review_id").limit(20)
)


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date
00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,null,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que seja trocado.",2018-03-07T00:00:00.000Z
00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,null,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que seja trocado.",2018-03-07T00:00:00.000Z
0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,null,null,2017-09-21T00:00:00.000Z
0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,null,null,2017-09-21T00:00:00.000Z
0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,null,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07T00:00:00.000Z
0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,null,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07T00:00:00.000Z
017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,null,null,2018-03-02T00:00:00.000Z
017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,null,null,2018-03-02T00:00:00.000Z
0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,null,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o status aparece como entregue. Solicito providências",2017-09-09T00:00:00.000Z
0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,null,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o status aparece como entregue. Solicito providências",2017-09-09T00:00:00.000Z


In [0]:

# Count all records
total_records = reviews_bronze.count()

# Count completely distinct rows
unique_records = reviews_bronze.dropDuplicates().count()

print("Total Bronze review records:", total_records)
print("Completely unique records:", unique_records)
print("Exact duplicate rows:", total_records - unique_records)


Total Bronze review records: 99224
Completely unique records: 99224
Exact duplicate rows: 0


In [0]:

# Load corrected Bronze Reviews table
reviews_bronze = spark.table("workspace.bronze.reviews_raw")

# Remove only completely identical rows
reviews_clean = reviews_bronze.dropDuplicates()

# Save corrected Silver table
(reviews_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.reviews_clean"))

print("Silver Reviews table rebuilt successfully!")

print("Bronze records:", reviews_bronze.count())
print("Silver records:", reviews_clean.count())


Silver Reviews table rebuilt successfully!
Bronze records: 99224
Silver records: 99224


In [0]:

from pyspark.sql.functions import col, count

# Find duplicate review_id + order_id combinations
duplicate_pairs = (
    reviews_clean
    .groupBy("review_id", "order_id")
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

print("Duplicate review-order pairs:", duplicate_pairs.count())

display(duplicate_pairs.limit(10))


Duplicate review-order pairs: 0


review_id,order_id,record_count


In [0]:

reviews = spark.table("workspace.silver.reviews_clean")
orders = spark.table("workspace.silver.orders_clean")

missing = reviews.filter(
    col("order_id").isNull()
).count()

unmatched = reviews.join(
    orders.select("order_id"),
    on="order_id",
    how="left_anti"
).count()

print("Total Silver reviews:", reviews.count())
print("Missing order IDs:", missing)
print("Unmatched order IDs:", unmatched)


Total Silver reviews: 99224
Missing order IDs: 0
Unmatched order IDs: 0
